In [1]:
from pyspark.sql import SparkSession

In [2]:
spark = SparkSession.builder.appName("Order Reviews Profiling").getOrCreate()

c:\Users\LENOVO\Desktop\AdvancedPySparkPipeline\.venv\Lib\site-packages\pyspark\testing\utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()


In [3]:
order_reviews = spark.read.parquet("../data/bronze/reviews")

In [4]:
order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [5]:
order_reviews.show(10,truncate = False)

+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message                                                                                                                                                        |review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+--------------------+-----------------------+
|7bc2406110b926393aa56f80a40eba40|

In [30]:
from pyspark.sql.functions import col,count,when,countDistinct

In [7]:
test_review_score = order_reviews.select(
    col("review_score").alias("score_test")
)

test_review_score.show(10)

+----------+
|score_test|
+----------+
|         4|
|         5|
|         5|
|         5|
|         5|
|         1|
|         5|
|         5|
|         5|
|         4|
+----------+
only showing top 10 rows


In [8]:
test_review_score.distinct().show()

+----------+
|score_test|
+----------+
|         1|
|         3|
|         5|
|         4|
|         2|
+----------+



In [17]:
order_reviews.select("review_score").distinct().show()

+------------+
|review_score|
+------------+
|           1|
|           3|
|           5|
|           4|
|           2|
+------------+



In [18]:
order_reviews.count()

99224

In [19]:
order_reviews.printSchema()

root
 |-- review_id: string (nullable = true)
 |-- order_id: string (nullable = true)
 |-- review_score: integer (nullable = true)
 |-- review_comment_title: string (nullable = true)
 |-- review_comment_message: string (nullable = true)
 |-- review_creation_date: timestamp (nullable = true)
 |-- review_answer_timestamp: timestamp (nullable = true)



In [20]:
order_reviews.select([
    count(when(col(c).isNull(),c)).alias(c)
    for c in order_reviews.columns
    ]).show(vertical=True)

-RECORD 0------------------------
 review_id               | 0     
 order_id                | 0     
 review_score            | 0     
 review_comment_title    | 87656 
 review_comment_message  | 58247 
 review_creation_date    | 0     
 review_answer_timestamp | 0     



In [21]:
order_reviews.select("review_score").show(20,truncate=False)

+------------+
|review_score|
+------------+
|4           |
|5           |
|5           |
|5           |
|5           |
|1           |
|5           |
|5           |
|5           |
|4           |
|5           |
|5           |
|4           |
|4           |
|3           |
|5           |
|2           |
|5           |
|3           |
|1           |
+------------+
only showing top 20 rows


In [24]:
duplicate_order_reviews = order_reviews.groupBy("review_id").count().filter(col("count")>1)
duplicate_order_reviews.show(truncate=False)
print("Duplicate Order Reviews", duplicate_order_reviews.count())

+--------------------------------+-----+
|review_id                       |count|
+--------------------------------+-----+
|f144ac1998474653203c861be02fd31f|2    |
|e4f5fbbbf2fa8f259e02e0e529eeae26|2    |
|868c4827b9ab7272338ae1c95d192505|2    |
|a8ffb30c74397c7cf37ab595d3c1662e|2    |
|f6e05a3f6dbc3bdb1ec3068725c67677|2    |
|c9a406d3076559e35f713ec82223d39c|2    |
|95cfb15ad3477fec574b561d1daea972|2    |
|dbcd301b59c7e85e61c10afd146e68ad|2    |
|9a07cb8136a1792e7984d827e2336d60|2    |
|dd43a798c9e8baf7b69297bb076c517d|2    |
|ef60daa08e71f279bffca277370f205f|2    |
|4812973f7fb59a488cb5fcbafe275581|2    |
|f45e3373e177f56944a549ff601f838b|2    |
|9d7b240dff5539983f5def389e2c1bf7|2    |
|4e9a141db068cf6856ce85ce8c3ef682|2    |
|c1dd7c6a2154caaf1a6accd68842286f|2    |
|bee3c76b6017ec9deaeef78bd5dbef21|2    |
|1961cafe1d1cbd4cc701c2f0160467ff|2    |
|d8a302244615111a0399d88db9916fd2|2    |
|fde5986d35c89aa1b6ce4149de82a0d3|2    |
+--------------------------------+-----+
only showing top

In [28]:
duplicate_id = "fde5986d35c89aa1b6ce4149de82a0d3"
order_reviews.filter(
    col("review_id") == duplicate_id
).show(truncate=False)

+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|review_id                       |order_id                        |review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|
+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----------------------+
|fde5986d35c89aa1b6ce4149de82a0d3|c892dec5d74e3ba28920ec30498fd00f|5           |NULL                |NULL                  |2018-07-24 00:00:00 |2018-07-25 14:22:14    |
|fde5986d35c89aa1b6ce4149de82a0d3|cba44aa41766f6210b875f576a4241e4|5           |NULL                |NULL                  |2018-07-24 00:00:00 |2018-07-25 14:22:14    |
+--------------------------------+--------------------------------+------------+--------------------+----------------------+--------------------+-----

In [31]:
duplicate_review_orders = (
    order_reviews
    .groupBy("review_id")
    .agg(
        countDistinct("order_id").alias("order_count")
    )
    .filter(col("order_count") > 1)
)

duplicate_review_orders.show()

+--------------------+-----------+
|           review_id|order_count|
+--------------------+-----------+
|f144ac19984746532...|          2|
|c9a406d3076559e35...|          2|
|dd43a798c9e8baf7b...|          2|
|9d7b240dff5539983...|          2|
|f45e3373e177f5694...|          2|
|59d160f8ce9217282...|          2|
|13d45c97ae78e0ef1...|          2|
|8b8f495d30fca09fd...|          2|
|e80e9b305e64fd907...|          2|
|4687cd3dc8cbaa6b8...|          2|
|5838f546d8d940f8e...|          2|
|3d0746e0f8bf4bd55...|          2|
|ad5b5b2868cf74d9d...|          2|
|5d1f78d752c653e01...|          2|
|2c6c08892b83ba4c1...|          2|
|d6d5c4f6a64c5fccf...|          2|
|73ee4a22d6266774e...|          2|
|0c1f10775ef729e60...|          2|
|034528cb00b6bc981...|          2|
|7d1a556b31dc7aac8...|          2|
+--------------------+-----------+
only showing top 20 rows


In [32]:
from pyspark.sql.functions import count

duplicate_rows = (
    order_reviews
    .groupBy(order_reviews.columns)
    .count()
    .filter(col("count") > 1)
)

duplicate_rows.show()

print("Exact duplicate rows:", duplicate_rows.count())

+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+
|review_id|order_id|review_score|review_comment_title|review_comment_message|review_creation_date|review_answer_timestamp|count|
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+
+---------+--------+------------+--------------------+----------------------+--------------------+-----------------------+-----+

Exact duplicate rows: 0
